# 🚀 Free Claude Code Backend via Google Colab (T4 GPU)
Run **Qwen 2.5 Coder** on a free 16GB T4 GPU in Google Colab and connect your local **Claude Code CLI** with zero cost and zero rate limits!

### 📌 Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run the code cell below.
3. Copy the generated `trycloudflare.com` URL and paste it into your local `litellm_config.yaml`.

In [ ]:
# 1-Click Setup: Ollama + Qwen 2.5 Coder + Cloudflare Tunnel
import os, subprocess, time, re

print("[1/5] Installing dependencies...")
!apt-get update -qq && apt-get install -y -qq zstd

print("\n[2/5] Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n[3/5] Installing Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("\n[4/5] Starting Ollama server with full remote access...")
!pkill -f "ollama serve"
!pkill -f "cloudflared"

os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0"
subprocess.Popen(["ollama", "serve"], env=os.environ)
time.sleep(3)

# Choose model: 'qwen2.5-coder:7b' (fast) or 'qwen2.5-coder:14b' (smarter)
MODEL_NAME = "qwen2.5-coder:7b"
print(f"\n[5/5] Downloading {MODEL_NAME} to GPU memory...")
!ollama pull {MODEL_NAME}

print("\n🌐 Starting Cloudflare Public Tunnel to Ollama...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "=" * 65)
print("🎉 YOUR FREE OLLAMA PUBLIC TUNNEL URL IS READY:")
print(f"👉 {public_url}")
print("=" * 65)
print("\nKeep this Colab tab open while using Claude Code!")